In [1]:
#Author: Logan Kline
    # GLBM
    #April 23, 2024
    #Feed in S123 files, search for site updates, then update feature class
    ######################################################################

# UPDATED v3 5.1.2024
#This version addresses AGOL issues with datetime conversions
# UPDATED v4 5.23.2024
#This version addresses includes edits to adapt to BOEM-CLB project, by Mike Bollinger
# UPDATED v5 6.11.2024
#This version actually works, by Mike Bollinger, Madeleine Gallop
import arcpy
import pandas as pd
from IPython.display import display
from arcgis.features import GeoAccessor, GeoSeriesAccessor, FeatureLayerCollection
from arcgis.gis import GIS
from arcgis import geometry
import os
import datetime as dt
from datetime import timedelta

sys.path.append(r"C:\Users\Mike.Bollinger\Documents\Python\VehicleTools") ##UPDATE## to Folder where "OverwiteFS.py is located"
import OverwriteFS

gis = GIS("home")
print("Logged in to " + str(gis.properties.name) + " as " + str(gis.properties.user.username))

C:\Program Files\ArcGIS\Pro\bin\Python\envs\arcgispro-py3\lib\site-packages\scipy\__init__.py:138: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.26.4)
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion} is required for this version of "


Logged in to NOAA GeoPlatform as mike.bollinger_noaa


In [2]:
#access s123 and save layers from AGOL
s123_cruise = gis.content.get('00d6b4deec944f3992c7a6ce016f3a7d').layers[0] #NCCOS UMS database cruise level layer, point
s123_auv = gis.content.get('00d6b4deec944f3992c7a6ce016f3a7d').layers[1] #NCCOS UMS AUV database point layer
s123_rov = gis.content.get('00d6b4deec944f3992c7a6ce016f3a7d').layers[2] #NCCOS UMS ROV database point layer
s123_dc = gis.content.get('00d6b4deec944f3992c7a6ce016f3a7d').layers[3] #NCCOS UMS Drop Camera database point layer

existingFSid = '8cdd28ad7790423089781259c8708034'
boem_samplesites = gis.content.get(existingFSid).layers[0] #sampling points already up in AGOL

CompletedCSVpath = r"C:\Users\Mike.Bollinger\Documents\R\NF-24-06_TEST\Outputs\SamplingPoints\Completed_Sampling_Points.csv"

#convert to databases
s123_cruise_dbf = pd.DataFrame(s123_cruise.query().sdf)
s123_auv_dbf = pd.DataFrame(s123_auv.query().sdf)
s123_rov_dbf = pd.DataFrame(s123_rov.query().sdf)
s123_dc_dbf = pd.DataFrame(s123_dc.query().sdf)
boem_samplesites_dbf = pd.DataFrame(boem_samplesites.query().sdf)


print(boem_samplesites_dbf)

    SiteID        Lon        Lat      Easting     Northing  SurveyPrio  \
0        1 -77.854471  33.402887   792557.791  3700376.285           3   
1        2 -77.839455  33.402505  793956.2123  3700376.285           3   
2        3  -77.82444  33.402122  795354.6337  3700376.285           6   
3        4 -77.809425  33.401736  796753.0551  3700376.285           6   
4        5 -77.884053  33.416241  789760.9482  3701774.707           4   
..     ...        ...        ...          ...          ...         ...   
91      92 -77.875235  33.497354     790310.5    3710797.5           1   
92      93 -77.869918  33.504278     790781.5    3711580.5           1   
93      94 -77.878384  33.473761     790096.5    3708171.5           1   
94      95 -77.875369  33.506471     790267.5    3711808.5           1   
95      96 -77.872386  33.512534     790524.5    3712489.5           1   

        Design DateGenerated DayNumber   AUV_Completed_Date  \
0      Regular    2024-06-07   DN24159          

In [3]:
boem_samplesites_dbf['AUV_Completed_Date'] = pd.to_datetime(boem_samplesites_dbf['AUV_Completed_Date'])
boem_samplesites_dbf['ROV_Completed_Date'] = pd.to_datetime(boem_samplesites_dbf['ROV_Completed_Date'])
boem_samplesites_dbf['DropCam_Completed_Date'] = pd.to_datetime(boem_samplesites_dbf['DropCam_Completed_Date'])
boem_samplesites_dbf['Most_Recent_Survey_Date'] = pd.to_datetime(boem_samplesites_dbf['Most_Recent_Survey_Date'])
print(boem_samplesites_dbf)

    SiteID        Lon        Lat      Easting     Northing  SurveyPrio  \
0        1 -77.854471  33.402887   792557.791  3700376.285           3   
1        2 -77.839455  33.402505  793956.2123  3700376.285           3   
2        3  -77.82444  33.402122  795354.6337  3700376.285           6   
3        4 -77.809425  33.401736  796753.0551  3700376.285           6   
4        5 -77.884053  33.416241  789760.9482  3701774.707           4   
..     ...        ...        ...          ...          ...         ...   
91      92 -77.875235  33.497354     790310.5    3710797.5           1   
92      93 -77.869918  33.504278     790781.5    3711580.5           1   
93      94 -77.878384  33.473761     790096.5    3708171.5           1   
94      95 -77.875369  33.506471     790267.5    3711808.5           1   
95      96 -77.872386  33.512534     790524.5    3712489.5           1   

        Design DateGenerated DayNumber  AUV_Completed_Date  \
0      Regular    2024-06-07   DN24159           

In [4]:
if not s123_cruise_dbf.empty:
    # Convert 'entryDay' column to datetime type
    s123_cruise_dbf['entryDay'] = pd.to_datetime(s123_cruise_dbf['entryDay'], errors='coerce')
    four_hours = timedelta(hours=4)
    s123_cruise_dbf['entryDay'] += four_hours

    for index, row in s123_cruise_dbf.iterrows():
        # Check if all specified columns contain NaN values
        if pd.isna(row['EODsuccessfulROV']) and pd.isna(row['EODsuccessfulAUV']) and pd.isna(row['EODsuccessfulDC']):
            continue  # Skip to next iteration if all specified columns contain NaN values

    # Handle non-string values in 'EODsuccessfulAUV' column
        if pd.notna(row['EODsuccessfulAUV']):
            # Extract integers from the string
            integers = [int(x.strip()) for x in str(row['EODsuccessfulAUV']).split(',') if x.strip().isdigit()]

            # Check if integers exist in boem_samplesites_dbf['SiteID']
            for integer in integers:
                if integer in boem_samplesites_dbf['SiteID'].values:
                    # Update corresponding ROV_Completed_Date with entryDay value
                    boem_samplesites_dbf.loc[boem_samplesites_dbf['SiteID'] == integer, 'AUV_Completed_Date'] = row['entryDay']

         # Handle non-string values in 'EODsuccessfulROV' column
        if pd.notna(row['EODsuccessfulROV']):
            # Extract integers from the string
            integers = [int(x.strip()) for x in str(row['EODsuccessfulROV']).split(',') if x.strip().isdigit()]

            # Check if integers exist in boem_samplesites_dbf['SiteID']
            for integer in integers:
                if integer in boem_samplesites_dbf['SiteID'].values:
                    # Update corresponding ROV_Completed_Date with entryDay value
                    boem_samplesites_dbf.loc[boem_samplesites_dbf['SiteID'] == integer, 'ROV_Completed_Date'] = row['entryDay']

        # Handle non-string values in 'EODsuccessfulDC' column
        if pd.notna(row['EODsuccessfulDC']):
            # Extract integers from the string
            integers = [int(x.strip()) for x in str(row['EODsuccessfulDC']).split(',') if x.strip().isdigit()]

            # Check if integers exist in boem_samplesites_dbf['SiteID']
            for integer in integers:
                if integer in boem_samplesites_dbf['SiteID'].values:
                    # Update corresponding DropCam_Completed_Date with entryDay value
                    boem_samplesites_dbf.loc[boem_samplesites_dbf['SiteID'] == integer, 'DropCam_Completed_Date'] = row['entryDay']

else:
    print("NOAA S123 empty, skipping")


In [5]:
print(boem_samplesites_dbf)

    SiteID        Lon        Lat      Easting     Northing  SurveyPrio  \
0        1 -77.854471  33.402887   792557.791  3700376.285           3   
1        2 -77.839455  33.402505  793956.2123  3700376.285           3   
2        3  -77.82444  33.402122  795354.6337  3700376.285           6   
3        4 -77.809425  33.401736  796753.0551  3700376.285           6   
4        5 -77.884053  33.416241  789760.9482  3701774.707           4   
..     ...        ...        ...          ...          ...         ...   
91      92 -77.875235  33.497354     790310.5    3710797.5           1   
92      93 -77.869918  33.504278     790781.5    3711580.5           1   
93      94 -77.878384  33.473761     790096.5    3708171.5           1   
94      95 -77.875369  33.506471     790267.5    3711808.5           1   
95      96 -77.872386  33.512534     790524.5    3712489.5           1   

        Design DateGenerated DayNumber  AUV_Completed_Date  \
0      Regular    2024-06-07   DN24159           

In [6]:

def check_status(row):
    if row['AUV_Completed_Date'] < pd.Timestamp(2024, 1, 1):
        return 'Incomplete'
    if row['ROV_Completed_Date'] < pd.Timestamp(2024, 1, 1):
        return 'Incomplete'
    if row['DropCam_Completed_Date'] < pd.Timestamp(2024, 1, 1):
        return 'Incomplete'
    if pd.isna(row['AUV_Completed_Date']) and pd.isna(row['ROV_Completed_Date']) and pd.isna(row['DropCam_Completed_Date']):
        return 'Incomplete'
    #if row['AUV'] == ' ' and row['ROV'] == ' ' and row['DropCam'] == ' ' and row['Ponar'] == ' ':
    #    return 'Extra'
    return 'Complete'

# Apply function to each row to update 'Status'
boem_samplesites_dbf['Status'] = boem_samplesites_dbf.apply(check_status, axis=1)

def check_incomplete(df, columns):
    # Check for NaT in specified columns
    if df[columns].isna().any().any():
        return "Incomplete"



# Columns to check
columns_to_check = ['AUV_Completed_Date', 'ROV_Completed_Date', 'DropCam_Completed_Date']

# Apply the function
result = check_incomplete(boem_samplesites_dbf, columns_to_check)
print(result)

print("Status updated")

Incomplete
Status updated


In [7]:
print(boem_samplesites_dbf)

    SiteID        Lon        Lat      Easting     Northing  SurveyPrio  \
0        1 -77.854471  33.402887   792557.791  3700376.285           3   
1        2 -77.839455  33.402505  793956.2123  3700376.285           3   
2        3  -77.82444  33.402122  795354.6337  3700376.285           6   
3        4 -77.809425  33.401736  796753.0551  3700376.285           6   
4        5 -77.884053  33.416241  789760.9482  3701774.707           4   
..     ...        ...        ...          ...          ...         ...   
91      92 -77.875235  33.497354     790310.5    3710797.5           1   
92      93 -77.869918  33.504278     790781.5    3711580.5           1   
93      94 -77.878384  33.473761     790096.5    3708171.5           1   
94      95 -77.875369  33.506471     790267.5    3711808.5           1   
95      96 -77.872386  33.512534     790524.5    3712489.5           1   

        Design DateGenerated DayNumber  AUV_Completed_Date  \
0      Regular    2024-06-07   DN24159           

In [8]:

#update Most_Recent_Survey_Date
for index, row in boem_samplesites_dbf.iterrows():
    # Extract dates from specified columns
    dates = [row['AUV_Completed_Date'], row['ROV_Completed_Date'], row['DropCam_Completed_Date']]

    # Filter out NaT values
    valid_dates = [date for date in dates if pd.notna(date)]

    # Check if there are any valid dates
    if valid_dates:
        # Get the maximum date
        most_recent_date = max(valid_dates)

        # Update 'Most_Recent_Survey_Date' column with the most recent date
        boem_samplesites_dbf.at[index, 'Most_Recent_Survey_Date'] = most_recent_date

print("Time updated successfully")
print(boem_samplesites_dbf)

Time updated successfully
    SiteID        Lon        Lat      Easting     Northing  SurveyPrio  \
0        1 -77.854471  33.402887   792557.791  3700376.285           3   
1        2 -77.839455  33.402505  793956.2123  3700376.285           3   
2        3  -77.82444  33.402122  795354.6337  3700376.285           6   
3        4 -77.809425  33.401736  796753.0551  3700376.285           6   
4        5 -77.884053  33.416241  789760.9482  3701774.707           4   
..     ...        ...        ...          ...          ...         ...   
91      92 -77.875235  33.497354     790310.5    3710797.5           1   
92      93 -77.869918  33.504278     790781.5    3711580.5           1   
93      94 -77.878384  33.473761     790096.5    3708171.5           1   
94      95 -77.875369  33.506471     790267.5    3711808.5           1   
95      96 -77.872386  33.512534     790524.5    3712489.5           1   

        Design DateGenerated DayNumber  AUV_Completed_Date  \
0      Regular    2024-

In [9]:
#Update CSV Version of points
boem_samplesites_dbf.to_csv(CompletedCSVpath)

In [10]:
print(boem_samplesites)

<FeatureLayer url:"https://services2.arcgis.com/C8EMgrsFcRFL6LrL/arcgis/rest/services/BOEM_CLB_Sample_Sites/FeatureServer/0">


In [11]:
boem_FLC = gis.content.get(existingFSid)
print(boem_FLC)

<Item title:"BOEM CLB Sample Sites" type:Feature Layer Collection owner:mike.bollinger_noaa>


In [12]:
#Bring in the original feature layer collection
boem_flayer_collection = FeatureLayerCollection.fromitem(boem_FLC)
#boem_flayer_collection = FeatureLayerCollection.fromitem(ssITEM)
boem_flayer_collection.properties.layers[0].name

'BOEM_CLB_Sample_Sites'

In [13]:
#the Overwrite
boem_flayer_collection.manager.overwrite(CompletedCSVpath)

{'success': True}